In [1]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd
from datetime import date

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [2]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [3]:
d = date.today().strftime("%Y%m%d")

In [4]:
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [5]:
wf_pts_export = r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels_pts'
gflu = r'E:\Tasks\REMM-Manage-Base-Year-Data-2\Inputs\gflu_2025.gdb\gflu_2025'
mag_open_space = r'E:\Tasks\REMM-Manage-Base-Year-Data-2\Inputs\MAG_Centers.gdb\MAG_MeetingPolygons_OpenSpaces_CoordUpdate'

# Zoning Baseline from GFLU

In [ ]:
# apply zoning baseline (GFLU)

# spatial join
target_features = wf_pts_export
join_features = gflu
output_features = os.path.join(gdb, 'parcel_gflu2025_join')

fieldmappings = arcpy.FieldMappings()
fieldmappings.addTable(target_features)
fieldmappings.addTable(join_features)

# land use type
fieldindex = fieldmappings.findFieldMapIndex('GenLUType2')
fieldmap = fieldmappings.getFieldMap(fieldindex)
fieldmap.mergeRule = 'First'
fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# max dua
fieldindex = fieldmappings.findFieldMapIndex('MaxDUA')
fieldmap = fieldmappings.getFieldMap(fieldindex)
fieldmap.mergeRule = 'Max'
fieldmappings.replaceFieldMap(fieldindex, fieldmap)

sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
gflu_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [ ]:
gflu_df.head()

# use max dua from gflu, leave no data as null or -9999
gflu_df = gflu_df.rename({'MaxDUA':'max_dua_new'}, axis=1)

# base max far = 0.5 in allowed in residential types, more in other types,  null in others
gflu_df['max_far_new'] = 0.5


#==========================
# gen lu 2 --> types(1-8) , add zero for for not allowed
#==========================

gflu_df['type1'] = 0
gflu_df['type2'] = 0
gflu_df['type3'] = 0
gflu_df['type4'] = 0
gflu_df['type5'] = 0
gflu_df['type6'] = 0
gflu_df['type7'] = 0
gflu_df['type8'] = 0

# Residential Single-Family --> 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Residential Single-Family', 'type1'] = 1

# Residential Multi-Family --> 1,2
gflu_df.loc[gflu_df['GenLUType2'] == 'Residential Multi-Family', 'type1'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Residential Multi-Family', 'type1'] = 1

# Any Residential --> 1,2
gflu_df.loc[gflu_df['GenLUType2'] == 'Any Residential', 'type1'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Any Residential', 'type1'] = 1

# Mixed Commercial --> 4,5,6,7,8
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Commercial', 'type4'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Commercial', 'type5'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Commercial', 'type6'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Commercial', 'type7'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Commercial', 'type8'] = 1

# Mixed Use --> 2,4,5,6,7,8
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Use', 'type2'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Use', 'type4'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Use', 'type5'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Use', 'type6'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Use', 'type7'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Mixed Use', 'type8'] = 1

# Industrial --> 3,4,5
gflu_df.loc[gflu_df['GenLUType2'] == 'Industrial', 'type3'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Industrial', 'type4'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Industrial', 'type5'] = 1

# Office -->  4,5
gflu_df.loc[gflu_df['GenLUType2'] == 'Office', 'type4'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Office', 'type5'] = 1

# Retail -->  4,5
gflu_df.loc[gflu_df['GenLUType2'] == 'Retail', 'type4'] = 1
gflu_df.loc[gflu_df['GenLUType2'] == 'Retail', 'type5'] = 1

# Government/Education --> 6
gflu_df.loc[gflu_df['GenLUType2'] == 'Government / Education', 'type6'] = 1

# Parks / Protected Lands / Agriculture --> 0,0,0,0,0,0,0,0
gflu_df.loc[gflu_df['GenLUType2'] == 'Parks / Protected Lands / Agriculture', 'type1'] = 0
gflu_df.loc[gflu_df['GenLUType2'] == 'Parks / Protected Lands / Agriculture', 'type2'] = 0
gflu_df.loc[gflu_df['GenLUType2'] == 'Parks / Protected Lands / Agriculture', 'type3'] = 0
gflu_df.loc[gflu_df['GenLUType2'] == 'Parks / Protected Lands / Agriculture', 'type4'] = 0
gflu_df.loc[gflu_df['GenLUType2'] == 'Parks / Protected Lands / Agriculture', 'type5'] = 0
gflu_df.loc[gflu_df['GenLUType2'] == 'Parks / Protected Lands / Agriculture', 'type6'] = 0
gflu_df.loc[gflu_df['GenLUType2'] == 'Parks / Protected Lands / Agriculture', 'type7'] = 0
gflu_df.loc[gflu_df['GenLUType2'] == 'Parks / Protected Lands / Agriculture', 'type8'] = 0

gflu_df = gflu_df[['parcel_id',  'max_far','max_dua', 'type1', 'type2', 'type3', 'type4', 'type5', 'type6','type7', 'type8']].copy()

In [ ]:
# mag_open_space

# spatial join
target_features = wf_pts_export
join_features = mag_open_space
output_features = os.path.join(gdb, 'parcel_magOpenSpace_join')

fieldmappings = arcpy.FieldMappings()
fieldmappings.addTable(target_features)
fieldmappings.addTable(join_features)

sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
magOpenSpace_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [ ]:
magOpenSpace_df.loc[magOpenSpace_df['Join_Count'] >= 1,'OpenSpaceMAG'] = 1
magOpenSpace_df.loc[(magOpenSpace_df['Join_Count'] < 1) | (magOpenSpace_df['Join_Count'].isna()==True),'OpenSpaceMAG'] = 0
magOpenSpace_df = magOpenSpace_df[['parcel_id', 'OpenSpaceMAG']].copy()

In [ ]:
new_zoning = gflu_df.merge(magOpenSpace_df, on='parcel_id', how='left')
new_zoning.to_csv(os.path.join(outputs[0], 'new_zoning_20260108.csv'), index=False)